In [1]:
from dotenv import load_dotenv
import os
import time
import json
import requests
import pandas as pd

# Ruta absoluta o relativa al .env en la raíz del repo
env_path = os.path.abspath(os.path.join(os.getcwd(), "../../..", ".env"))
load_dotenv(dotenv_path=env_path)

# Obtener clave de OpenAI desde .env
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY:
    print("Clave de OpenAI cargada correctamente.")
else:
    raise ValueError("No se encontró la clave de OpenAI. Verifica la ruta del .env.")

Clave de OpenAI cargada correctamente.


In [2]:
def call_gpt_api(prompt, text):
    """
    Sends a text and a prompt to GPT-4o and returns the generated summary.
    """
    url = "https://api.openai.com/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "gpt-4o",
        "messages": [
            {"role": "system", "content": "You are an expert assistant specialized in simplifying biomedical language."},
            {"role": "user", "content": f"{prompt}\n\nText:\n{text}"}
        ],
        "max_tokens": 1024,
        "temperature": 0.7
    }

    try:
        start_time = time.time()
        response = requests.post(url, headers=headers, json=payload)
        elapsed = time.time() - start_time

        print(f"Status code: {response.status_code}")
        if response.status_code == 200:
            data = response.json()
            print("Response keys:", data.keys())
            output = data["choices"][0]["message"]["content"]
            return output.strip(), elapsed
        else:
            print(f"HTTP error {response.status_code}: {response.text[:200]}")
            return None, elapsed

    except Exception as e:
        print("Request error:", e)
        return None, None

In [3]:
prompt = "Summarize this biomedical paragraph in plain language for the public."
text = "The administration of statins has been shown to reduce LDL cholesterol and cardiovascular risk."
resumen, tiempo = call_gpt_api(prompt, text)
print(resumen)
print(f"Time: {tiempo:.2f} s")

Status code: 200
Response keys: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'service_tier', 'system_fingerprint'])
Taking statins can lower "bad" cholesterol (LDL) levels and decrease the chances of heart-related problems.
Time: 2.83 s


In [4]:
ruta_dataset = "../data-sources/pre-processed/data_finetuning_test.csv"
df = pd.read_csv(ruta_dataset, encoding="utf-8", on_bad_lines="skip")
display(df.head(2))

,name,article,summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...


In [ ]:
# Preparar columna para resúmenes generados
df["gen_summary"] = ""

# Prompt para simplificar artículos
prompt = "Summarize this biomedical paragraph in plain language for the public."

# Ruta donde se guardará el CSV con los resultados
ruta_csv = "./results_gpt.csv"

# Procesar cada artículo
for i, fila in df.iterrows():
    print(f"\nProcessing {fila['name']} ({i+1}/{len(df)})...\n")
    resumen, tiempo = call_gpt_api(prompt, fila["article"])
    
    if resumen:
        df.loc[i, "gen_summary"] = resumen
        print(f"Response time: {tiempo:.2f} s\n")
    else:
        print("No response received.\n")

# Guardar resultados
df.to_csv(ruta_csv, index=False, encoding="utf-8")
print(f"Results saved to: {os.path.abspath(ruta_csv)}")
display(df.head())


📄 Processing 10.1002-14651858.CD009781.pub2 (1/380)...

Status code: 200
Response keys: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'service_tier', 'system_fingerprint'])
⏱️ Response time: 5.05 s


📄 Processing 10.1002-14651858.CD010694.pub2 (2/380)...

Status code: 200
Response keys: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'service_tier', 'system_fingerprint'])
⏱️ Response time: 4.59 s


📄 Processing 10.1002-14651858.CD009416.pub2 (3/380)...

Status code: 200
Response keys: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'service_tier', 'system_fingerprint'])
⏱️ Response time: 4.86 s


📄 Processing 10.1002-14651858.CD004104.pub4 (4/380)...

Status code: 200
Response keys: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'service_tier', 'system_fingerprint'])
⏱️ Response time: 4.58 s


📄 Processing 10.1002-14651858.CD012689.pub2 (5/380)...

Status code: 200
Response keys: dict_keys(['id', 'object', 

,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,Background:\nEye injuries like scratches on th...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,Venous leg ulcers are long-lasting wounds caus...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,Complex regional pain syndrome (CRPS) is a con...
3,10.1002-14651858.CD004104.pub4,Background\r\nNon‐invasive ventilation (NIV) w...,Non‐invasive ventilation for people with respi...,This study looked at a treatment called non-in...
4,10.1002-14651858.CD012689.pub2,Background\r\nSpace spraying is the dispersal ...,Insecticide space spraying for preventing mala...,Space spraying involves spraying insecticides ...


In [6]:
# Ruta del archivo csv guardado para verificar su contenido.
ruta_csv = "./results_gpt.csv"
df_check = pd.read_csv(ruta_csv)

df_check.info()
df_check.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         380 non-null    object
 1   article      380 non-null    object
 2   summary      380 non-null    object
 3   gen_summary  380 non-null    object
dtypes: object(4)
memory usage: 12.0+ KB


,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,Background:\nEye injuries like scratches on th...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,Venous leg ulcers are long-lasting wounds caus...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,Complex regional pain syndrome (CRPS) is a con...
3,10.1002-14651858.CD004104.pub4,Background\r\nNon‐invasive ventilation (NIV) w...,Non‐invasive ventilation for people with respi...,This study looked at a treatment called non-in...
4,10.1002-14651858.CD012689.pub2,Background\r\nSpace spraying is the dispersal ...,Insecticide space spraying for preventing mala...,Space spraying involves spraying insecticides ...


In [7]:
# Filtrar la fila faltante
faltante = df_check[df_check["gen_summary"].isnull()]
print(f"🔍 Faltan {len(faltante)} resúmenes.")
display(faltante[["name", "article"]])

🔍 Faltan 0 resúmenes.


,name,article


In [ ]:
# Reprocesar solo esa fila
if not faltante.empty:
    for i, fila in faltante.iterrows():
        print(f"\nReprocesando {fila['name']}...\n")
        resumen, tiempo = call_gpt_api(prompt, fila['article'])
        df_check.loc[i, "gen_summary"] = resumen if resumen else ""
        print(f"Reparado en {tiempo:.2f} s")

# Guardar nuevamente el CSV actualizado
df_check.to_csv(ruta_csv, index=False, encoding="utf-8")
print(f"Archivo actualizado: {os.path.abspath(ruta_csv)}")

✅ Archivo actualizado: c:\Users\maaro\OneDrive\Documentos\MaestriaIA\Despliegue de soluciones\Test\Proyecto-PLN-FLAG\src\api_tests\results_gpt.csv
